# IoTGuard LR + Dummy â€” Smoke pipeline and resource check

Training takes place on Kaggle only. Add the original G3B processed-data Dataset and the code bundle as Inputs. The smoke uses diagnostic subsets and produces no official comparison result.

The data package is not included in the code ZIP. Its manifest hash is pinned. No TEST model evaluation is performed. Consult the bundle README for source provenance, resource limits and output review.


## 1 Configure the Kaggle run

In [ ]:
# Change only input paths when necessary. Baseline settings are fixed for this run.
MODE = "smoke"
PACKAGE_DIRECTORY = None  # Automatic search for the pinned manifest under /kaggle/input.
THREADS = 4
CHUNK_SIZE = 8192
MAX_ITER = 1000

## 2 Load and verify the code bundle

In [ ]:
import hashlib
import importlib
import json
import stat
import sys
import tempfile
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
if not INPUT_ROOT.is_dir():
    raise RuntimeError("Import this notebook into Kaggle; do not run training locally.")

# Find an extracted bundle, or extract the small code ZIP (not the 6.58 GB data).
candidates = []
for marker in INPUT_ROOT.rglob("bundle_manifest.json"):
    try:
        identity = json.loads(marker.read_text())
    except (OSError, ValueError):
        continue
    if identity.get("bundle_id") == "iotguard_primary_lr_kaggle_v2":
        candidates.append(marker.parent)
if not candidates:
    archives = list(INPUT_ROOT.rglob("IoTGuard_Kaggle_LR_Bundle.zip"))
    if len(archives) != 1:
        raise RuntimeError("Add exactly one code-bundle Dataset to this notebook.")
    destination = Path(tempfile.mkdtemp(prefix="iotguard_code_bundle_v2_", dir="/kaggle/working"))
    with zipfile.ZipFile(archives[0]) as archive:
        if sum(m.file_size for m in archive.infolist()) > 10 * 1024**2:
            raise ValueError("Code bundle exceeds 10 MiB; do not extract a data ZIP here")
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if (
                not target.is_relative_to(destination.resolve())
                or "\\" in member.filename
                or ":" in member.filename
                or stat.S_ISLNK(member.external_attr >> 16)
            ):
                raise ValueError("Invalid bundle archive path")
        archive.extractall(destination)
    candidates = [destination]
if len(candidates) > 1:
    hashes = {
        hashlib.sha256((p / "bundle_manifest.json").read_bytes()).hexdigest() for p in candidates
    }
    if len(hashes) != 1:
        raise RuntimeError("Different code-bundle versions attached. Keep one release.")
    # Saved smoke outputs also contain an identical source snapshot. Deduplicate it.
    candidates = sorted(candidates, key=lambda p: ("source" in p.parts, str(p)))[:1]
BUNDLE_ROOT = candidates[0].resolve()
sys.path.insert(0, str(BUNDLE_ROOT))
runner = importlib.import_module("kaggle_lr_runner")
if Path(runner.__file__).resolve().parent != BUNDLE_ROOT:
    raise RuntimeError("Runner from another bundle is cached. Restart the Kaggle session.")
runner.check_bundle()
print("Source bundle:", BUNDLE_ROOT)
print("Runtime:", json.dumps(runner.environment(), indent=2))

## 3 Locate data and check prerequisites

In [ ]:
package_directory = runner.find_package(PACKAGE_DIRECTORY)
print("Pinned data package:", package_directory)
if MODE == "full":
    print(
        "Smoke prerequisite:",
        json.dumps(
            runner.smoke_receipt(
                runner.baseline_configuration(
                    max_iter=MAX_ITER, threads=THREADS, chunk_size=CHUNK_SIZE
                )
            ),
            indent=2,
        ),
    )

## 4 Train and evaluate on Kaggle

In [ ]:
status = runner.run(
    mode=MODE,
    package_directory=str(package_directory),
    smoke_train=50000,
    smoke_validation=10000,
    max_iter=MAX_ITER,
    threads=THREADS,
    chunk_size=CHUNK_SIZE,
)
print(json.dumps(status, indent=2))

## 5 Inspect and export results

Save Version with outputs, then download the result ZIP. A smoke run must not be reported as a full baseline.

In [ ]:
from IPython.display import FileLink, display

output_directory = Path(status["output_directory"])
expected = "SMOKE_PASS" if MODE == "smoke" else "BASELINES_COMPLETE"
if status["status"] != expected:
    print("NOT COMPLETE:", status["status"], "â€” send logs/status for review.")
else:
    print("Completed:", expected)
comparison = output_directory / "comparison.csv"
if comparison.exists():
    print(comparison.read_text())
display(FileLink(str(output_directory.with_suffix(".zip"))))
display(FileLink(str(output_directory / "run_status.json")))